<!-- # Toeplitz-hashing extractor による乱数抽出 -->

# Randomness extraction with the Toeplitz-hashing extractor

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 量子乱数生成器（QRNG）の raw data から、**Toeplitz-hashing extractor** で一様乱数を取り出します。手順は次の論文の Sec. III B にしたがいます。 -->

We extract uniform random numbers from the raw data of a quantum random number generator (QRNG) with the **Toeplitz-hashing extractor**. The procedure follows Sec. III B of the following paper.

> X. Ma, F. Xu, H. Xu, X. Tan, B. Qi, and H.-K. Lo, "Postprocessing for quantum random-number generators: Entropy evaluation and randomness extraction," *Phys. Rev. A* **87**, 062327 (2013); arXiv:1207.1473.

<!--
1. 入力のビット列を用意し、1 ビットあたりの min-entropy $h$ を決める（2.）
2. 出力の長さ $m$ を決める（3.）
3. シードから Toeplitz 行列を作り、FFT で入力との積を計算する（4.・5.）
4. 入力全体、またはブロックごとに抽出する（6.）
5. 抽出結果を保存する（7.）
-->

1. Prepare the input bit string and determine the min-entropy per bit $h$ (2.)
2. Determine the output length $m$ (3.)
3. Build the Toeplitz matrix from the seed and compute its product with the input using the FFT (4., 5.)
4. Extract from the whole input or block by block (6.)
5. Save the extraction results (7.)

---

<!-- ## 手法 -->

## Methods

### A. Toeplitz-hashing extractor

<!-- $n$ ビットの入力 $x$（行ベクトル）に $n \times m$ の Toeplitz 行列 $T$ を掛け、$m$ ビットを出力します。 -->

The $n$-bit input $x$ (a row vector) is multiplied by an $n \times m$ Toeplitz matrix $T$ to output $m$ bits.

$$
y = x\,T \bmod 2, \qquad T_{ij} = t_{i-j} \quad (0 \le i \le n-1,\ 0 \le j \le m-1)
$$

<!-- $T$ は各対角線上の成分が等しい行列で、$t_{-(m-1)}, \ldots, t_{n-1}$ の $n + m - 1$ ビットのシード $s$ で決まります。このノートブックでは、1 列目 $c$ と 1 行目 $r$ を -->

$T$ is a matrix whose entries are equal along each diagonal, and it is determined by an $(n + m - 1)$-bit seed $s$, namely $t_{-(m-1)}, \ldots, t_{n-1}$. In this notebook the first column $c$ and the first row $r$ are

$$
c = (t_0, \ldots, t_{n-1}) = (s_0, \ldots, s_{n-1}), \qquad
r = (t_0, t_{-1}, \ldots, t_{-(m-1)}) = (s_0, s_n, \ldots, s_{n+m-2})
$$

<!-- とします。シードを一様に選ぶと、Toeplitz 行列の族は 2-universal なハッシュ関数の族です（異なる $x \ne x'$ について $\Pr_T[xT = x'T] = 2^{-m}$）。 -->

When the seed is chosen uniformly, the family of Toeplitz matrices is a 2-universal family of hash functions ($\Pr_T[xT = x'T] = 2^{-m}$ for any $x \ne x'$).

<!-- ### B. 出力の長さ：Leftover Hash Lemma -->

### B. Output length: the Leftover Hash Lemma

<!-- 入力の min-entropy が $k$ 以上なら、2-universal なハッシュ関数による出力とシードの組は、一様乱数に $\varepsilon = 2^{(m-k)/2}$ まで近い（$\varepsilon$-close）強い抽出器になります（Ma の Lemma III.1）。これを $m$ について解き、整数に切り捨てて出力の長さにします。 -->

If the min-entropy of the input is at least $k$, the pair of the output of a 2-universal hash function and the seed is a strong extractor whose output is $\varepsilon$-close to uniform with $\varepsilon = 2^{(m-k)/2}$ (Lemma III.1 of Ma). Solving this for $m$ and truncating to an integer gives the output length.

$$
m = \left\lfloor k + 2 \log_2 \varepsilon \right\rfloor, \qquad k = nh
$$

<!-- $\varepsilon = 2^{-100}$ なら、出力は入力の min-entropy より 200 ビット短くなります。 -->

For $\varepsilon = 2^{-100}$, the output is 200 bits shorter than the min-entropy of the input.

<!-- ### C. FFT による計算 -->

### C. Computation with the FFT

<!-- $y_j = \sum_i x_i\, t_{i-j}$ は畳み込みなので、長さ $L = n + m - 1$ の巡回畳み込みで計算できます。 -->

Since $y_j = \sum_i x_i\, t_{i-j}$ is a convolution, it can be computed as a cyclic convolution of length $L = n + m - 1$.

$$
a = (r_0, \ldots, r_{m-1},\ c_{n-1}, c_{n-2}, \ldots, c_1), \qquad
\tilde{x} = (x_0, \ldots, x_{n-1}, \underbrace{0, \ldots, 0}_{m-1})
$$

$$
y_j = \Bigl[\mathrm{IFFT}\bigl(\mathrm{FFT}(a) \cdot \mathrm{FFT}(\tilde{x})\bigr)_j\Bigr] \bmod 2 \qquad (0 \le j \le m-1)
$$

<!-- $[\cdot]$ は最も近い整数への丸めです。計算量は $nm$ から $O(L \log L)$ になり、$n = 10^6$ でも 1 秒程度で計算できます。 -->

$[\cdot]$ denotes rounding to the nearest integer. The cost drops from $nm$ to $O(L \log L)$, and even $n = 10^6$ takes about one second.

<!-- ### D. ブロックに分けてシードを使い回す -->

### D. Splitting into blocks and reusing the seed

<!-- シード（$n + m - 1$ ビット）は出力（$m$ ビット）より長いので、1 回しか使わないと、使った一様乱数より得た一様乱数の方が少なくなります。Toeplitz-hashing extractor は強い抽出器なので、入力を $n_b$ ビットのブロックに分け、同じシードをすべてのブロックに使えます。 -->

The seed ($n + m - 1$ bits) is longer than the output ($m$ bits), so if it is used only once, we get fewer uniform random bits than we put in. Because the Toeplitz-hashing extractor is a strong extractor, the input can be split into blocks of $n_b$ bits and the same seed can be used for all blocks.

<!--
| 量 | 値 |
|---|---|
| 1 ブロックの出力 | $m_b = \lfloor n_b h + 2\log_2 \varepsilon \rfloor$ |
| 出力の合計 | $N_b\, m_b$（$N_b$ はブロック数） |
| シード | $n_b + m_b - 1$ ビット（1 回だけ用意する） |
| 全体の誤差 | $N_b\, \varepsilon$ |
-->

| Quantity | Value |
|---|---|
| Output per block | $m_b = \lfloor n_b h + 2\log_2 \varepsilon \rfloor$ |
| Total output | $N_b\, m_b$ ($N_b$ is the number of blocks) |
| Seed | $n_b + m_b - 1$ bits (prepared only once) |
| Total error | $N_b\, \varepsilon$ |

<!-- Ma はこの方法で $n_b = 4096$ のブロックを処理しています。 -->

Ma processes blocks of $n_b = 4096$ in this way.

<!-- ### 但し書き -->

### Caveats

<!--
- **$h$ の決め方**：$h$ は SP 800-90B の評価（`SP800_90B_textbook.ipynb`）で求めた値を `H_MIN_OVERRIDE` に入れると、さらに本番に近い実験検証になります。今回のノートブックでは Toeplitz-hashing extractor のプログラム動作の確認のみを目的とするため、特に $h$ の値を指定しない場合は、最も基本的な評価手法である MCV 推定（2.）を用います。
- **シード**：raw data とは独立な一様乱数でなければなりません。`SEED_RNG` に整数を指定すると、誰でも再現できる疑似乱数になるので、確認用にだけ使ってください。強い抽出器なので、シードを公開しても安全性は損なわれません。
- **量子的な side information**：攻撃者が raw data と相関のある量子系 $E$ を持つ場合も、$k$ を条件付き min-entropy $H_{\min}(X|E)$ にすれば B と同じ式が成り立ちます（M. Tomamichel *et al.*, *IEEE Trans. Inf. Theory* **57**, 5524 (2011)）。
- **Ma の式 (9)**：論文の式 (9) は $m = k - 2\log\varepsilon$ と印刷されていますが、Lemma III.1 と論文の数値例（$n = 4096$、$k \approx 3430$、$m = 3230$、$\varepsilon < 2^{-100}$）に合うのは B の $m = k + 2\log_2\varepsilon$ です。
- **丸めの誤差**：FFT は浮動小数点数で計算するので、丸める前の値が整数から 0.25 以上ずれた場合はエラーにします（$n = 10^6$ でもずれは $10^{-8}$ 程度です）。
-->

- **How to choose $h$**: Setting `H_MIN_OVERRIDE` to the value obtained from the SP 800-90B assessment (`SP800_90B_textbook.ipynb`) makes the experiment closer to a real deployment. Because this notebook is intended only to check that the Toeplitz-hashing extractor program works, the most basic assessment method, the MCV estimate (2.), is used when no value of $h$ is specified.
- **Seed**: It must be uniform random bits independent of the raw data. Setting `SEED_RNG` to an integer gives pseudorandom numbers that anyone can reproduce, so use it only for checking. Because this is a strong extractor, publishing the seed does not compromise security.
- **Quantum side information**: Even when an adversary holds a quantum system $E$ correlated with the raw data, the same formula as in B holds with $k$ replaced by the conditional min-entropy $H_{\min}(X|E)$ (M. Tomamichel *et al.*, *IEEE Trans. Inf. Theory* **57**, 5524 (2011)).
- **Ma's Eq. (9)**: Eq. (9) of the paper is printed as $m = k - 2\log\varepsilon$, but what agrees with Lemma III.1 and with the paper's numerical example ($n = 4096$, $k \approx 3430$, $m = 3230$, $\varepsilon < 2^{-100}$) is $m = k + 2\log_2\varepsilon$ in B.
- **Rounding error**: The FFT is computed in floating point, so an error is raised if a value before rounding deviates from an integer by 0.25 or more (even for $n = 10^6$ the deviation is about $10^{-8}$).

---

<!-- ## 使い方 -->

## How to use this notebook

<!-- 上のセルから順に実行してください。条件は、次の変数を書き換えて選びます。 -->

Run the cells in order from the top. The conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 既定値 |
|---|---|---|
| `N_BITS` | シミュレーションで作る入力のビット数 $n$ | `1_000_000` |
| `P1_GIVEN_0`, `P0_GIVEN_1` | シミュレーションの読み出し誤差 $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.02`, `0.04` |
| `SIM_SEED` | シミュレーションの乱数シード（`None` なら毎回違う結果） | `1` |
| `INPUT_PATH` | raw data のファイル（`None` ならシミュレーションを使う） | `None` |
| `H_MIN_OVERRIDE` | 1 ビットあたりの min-entropy $h$（`None` なら MCV 推定値） | `None` |
| `EPSILON` | 1 回の抽出（入力全体、または 1 ブロック）の誤差 $\varepsilon$ | `2.0 ** -100` |
| `SEED_RNG` | シードの作り方（`None` なら OS の暗号用乱数、整数なら再現できる疑似乱数） | `None` |
| `BLOCK_SIZE` | 1 ブロックのビット数 $n_b$（`None` なら入力全体を 1 つの行列で抽出する） | `None` |
| `SAVE_RESULTS` | 結果を保存するか（`True` なら `results/toeplitz/<日時>/` に保存） | `False` |
-->

| Variable | Meaning | Default |
|---|---|---|
| `N_BITS` | Number of input bits $n$ generated by the simulation | `1_000_000` |
| `P1_GIVEN_0`, `P0_GIVEN_1` | Readout errors of the simulation $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.02`, `0.04` |
| `SIM_SEED` | Random seed of the simulation (`None` gives different results each time) | `1` |
| `INPUT_PATH` | Raw-data file (`None` means use the simulation) | `None` |
| `H_MIN_OVERRIDE` | Min-entropy per bit $h$ (`None` means the MCV estimate) | `None` |
| `EPSILON` | Error $\varepsilon$ of one extraction (the whole input or one block) | `2.0 ** -100` |
| `SEED_RNG` | How the seed is generated (`None`: the OS's cryptographic random numbers; integer: reproducible pseudorandom numbers) | `None` |
| `BLOCK_SIZE` | Number of bits per block $n_b$ (`None` means extract the whole input with one matrix) | `None` |
| `SAVE_RESULTS` | Whether to save the results (`True` saves them in `results/toeplitz/<date_time>/`) | `False` |

<!-- ### 保存されるファイル -->

### Saved files

<!-- 1. の準備のセルで `SAVE_RESULTS = True` にすると、そのセルを実行した日時の名前のフォルダ `results/toeplitz/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作り、7. のセルで次のファイルを保存します。`False`（既定）では何も保存しません。 -->

If you set `SAVE_RESULTS = True` in the setup cell in 1., a folder named after the date and time that cell was run, `results/toeplitz/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created, and the following files are saved in it by the cell in 7. With `False` (the default), nothing is saved.

<!--
```
results/toeplitz/
└── 2026-11-10_153012/          RUN_DIR（1. のセルを実行した日時）
    ├── input_bits.txt             抽出の入力のビット列
    ├── toeplitz_extracted.txt      抽出したビット列
    ├── toeplitz_seed.txt           シード
    └── toeplitz_info.json          パラメータと入力の情報
```
-->

```
results/toeplitz/
└── 2026-11-10_153012/          RUN_DIR (date and time the cell in 1. was run)
    ├── input_bits.txt             input bit string of the extraction
    ├── toeplitz_extracted.txt      extracted bit string
    ├── toeplitz_seed.txt           seed
    └── toeplitz_info.json          parameters and information about the input
```

<!--
| ファイル | 形式 | 中身 |
|---|---|---|
| `input_bits.txt` | テキスト。`'0'` と `'1'` の文字を改行なしで並べる（1 文字 = 1 ビット） | 抽出に使った入力のビット列（シミュレーションでも `INPUT_PATH` のファイルでも保存する） |
| `toeplitz_extracted.txt` | 同じ形式 | 抽出したビット列。`load_bits` で読める。SP800_22 のノートブックの `BITS_FILE` に指定できる |
| `toeplitz_seed.txt` | 同じ形式 | Toeplitz 行列のシード（入力全体で抽出した場合は $n + m - 1$ ビット、ブロックごとの場合は $n_b + m_b - 1$ ビット） |
| `toeplitz_info.json` | JSON | 6.1 の戻り値 `result` の内容（主なキーは 6.1 の表）に、`input_path`, `simulation`, `h_min_source`, `block_size` を加えたもの。`input_path` は `INPUT_PATH` の値（シミュレーションなら `null`）、`simulation` はシミュレーションの条件（`N_BITS`, `P1_GIVEN_0`, `P0_GIVEN_1`, `SIM_SEED`）、`h_min_source` は `"MCV"` か `"H_MIN_OVERRIDE"` |
-->

| File | Format | Contents |
|---|---|---|
| `input_bits.txt` | Text. The characters `'0'` and `'1'` with no line breaks (1 character = 1 bit) | The input bit string of the extraction (saved both for the simulation and for an `INPUT_PATH` file) |
| `toeplitz_extracted.txt` | Same format | The extracted bit string. Can be read with `load_bits`, and can be given as `BITS_FILE` in the SP800_22 notebook |
| `toeplitz_seed.txt` | Same format | The seed of the Toeplitz matrix ($n + m - 1$ bits when extracting from the whole input; $n_b + m_b - 1$ bits when extracting block by block) |
| `toeplitz_info.json` | JSON | The contents of `result` returned in 6.1 (main keys in the table in 6.1), plus `input_path`, `simulation`, `h_min_source`, and `block_size`. `input_path` is the value of `INPUT_PATH` (`null` for the simulation), `simulation` holds the simulation conditions (`N_BITS`, `P1_GIVEN_0`, `P0_GIVEN_1`, `SIM_SEED`), and `h_min_source` is `"MCV"` or `"H_MIN_OVERRIDE"` |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま 7. のセルをもう一度実行するとエラー（`FileExistsError`）になるので、1. の準備のセルを実行し直して新しいフォルダを作ってください。
- 入力・シード・パラメータがそろっているので、`input_bits.txt` と `toeplitz_seed.txt` を同じ設定で抽出の関数に渡すと、同じ出力が得られます。
-->

- Files are never overwritten. If you run the cell in 7. again with the same `RUN_DIR`, an error (`FileExistsError`) is raised, so rerun the setup cell in 1. to create a new folder.
- Because the input, the seed, and the parameters are all saved, passing `input_bits.txt` and `toeplitz_seed.txt` to the extraction function with the same settings gives the same output.

<!-- ### 関数の家系図 -->

### Function call tree

<!-- このノートブックの関数の呼び出し関係です。上の（字下げの浅い）関数が、そのすぐ下に枝で続く関数を内部で呼び出します。 -->

This shows how the functions in this notebook call each other. A function higher up (less indented) internally calls the functions on the branches directly below it.

<!--
- `[5.]` のような右の番号は、**その関数が定義されている章**です。
- `← 6.2 のセルから実行` は、その章のコードセルから直接呼び出していることを示します。
- `（↑ 上で展開済み）` は、同じ関数の中身をすでに上の図で展開していることを示します。
-->

- The number on the right, such as `[5.]`, is **the section in which the function is defined**.
- `← run from the cell in 6.2` means the function is called directly from the code cell of that section.
- `(↑ expanded above)` means the contents of the same function have already been expanded earlier in the tree.

<!-- **抽出の処理** -->

**Extraction**

<!--
```
toeplitz_extraction             [6.1]  ← 6.2 のセルから実行
├── calculate_output_length     [3.]
└── toeplitz_hash_fft           [5.]
    ├── generate_toeplitz_seed  [4.]
    └── split_toeplitz_seed     [4.]

toeplitz_extraction_blocks      [6.1]  ← 6.2 のセルから実行
├── calculate_output_length     [3.]
├── generate_toeplitz_seed      [4.]
└── toeplitz_hash_fft           [5.]  （↑ 上で展開済み）
```
-->

```
toeplitz_extraction             [6.1]  ← run from the cell in 6.2
├── calculate_output_length     [3.]
└── toeplitz_hash_fft           [5.]
    ├── generate_toeplitz_seed  [4.]
    └── split_toeplitz_seed     [4.]

toeplitz_extraction_blocks      [6.1]  ← run from the cell in 6.2
├── calculate_output_length     [3.]
├── generate_toeplitz_seed      [4.]
└── toeplitz_hash_fft           [5.]  (↑ expanded above)
```

<!-- **入力の準備・評価・保存（他の関数を呼び出さず、セルから単独で使う関数）** -->

**Preparing, assessing, and saving the input (functions used on their own from a cell, without calling other functions)**

<!--
```
simulate_c1_bits  [2.]  ← 2. のセルから実行
load_bits         [2.]  ← 2. のセルから実行
mcv_min_entropy   [2.]  ← 2. のセルから実行
save_bits         [2.]  ← 7. のセルから実行
```
-->

```
simulate_c1_bits  [2.]  ← run from the cell in 2.
load_bits         [2.]  ← run from the cell in 2.
mcv_min_entropy   [2.]  ← run from the cell in 2.
save_bits         [2.]  ← run from the cell in 7.
```

<!-- **章ごとの定義の一覧** -->

**Functions defined in each section**

<!--
| 章 | 章の題 | 定義されている関数 |
|---|---|---|
| 2. | 入力のビット列を用意する | `simulate_c1_bits`、`load_bits`、`save_bits`、`mcv_min_entropy` |
| 3. | 出力の長さを決める | `calculate_output_length` |
| 4. | シードを作る | `generate_toeplitz_seed`、`split_toeplitz_seed` |
| 5. | FFT で Toeplitz 行列との積を計算する | `toeplitz_hash_fft` |
| 6.1 | 抽出の関数 | `toeplitz_extraction`、`toeplitz_extraction_blocks` |
-->

| Section | Section title | Functions defined |
|---|---|---|
| 2. | Preparing the input bit string | `simulate_c1_bits`, `load_bits`, `save_bits`, `mcv_min_entropy` |
| 3. | Determining the output length | `calculate_output_length` |
| 4. | Generating the seed | `generate_toeplitz_seed`, `split_toeplitz_seed` |
| 5. | Computing the product with the Toeplitz matrix using the FFT | `toeplitz_hash_fft` |
| 6.1 | Extraction functions | `toeplitz_extraction`, `toeplitz_extraction_blocks` |

<!-- ## 1. 準備 -->

## 1. Setup

<!-- ライブラリを読み込み、誤差 `EPSILON` とシードの作り方 `SEED_RNG` を決めます。 -->

We load the libraries and set the error `EPSILON` and how the seed is generated, `SEED_RNG`.

<!-- 結果の保存：`SAVE_RESULTS` で、7. で入力のビット列と抽出結果を保存するかどうかを選びます。`True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/toeplitz/2026-11-10_153012/`）に保存します。`False`（既定）では何も保存しません。 -->

Saving the results: `SAVE_RESULTS` selects whether the input bit string and the extraction results are saved in 7. With `True`, they are saved in a folder named after the date and time this cell was run (e.g., `results/toeplitz/2026-11-10_153012/`). With `False` (the default), nothing is saved.

In [ ]:
import json
import math
import secrets
import time
from datetime import datetime
from pathlib import Path

import numpy as np

# Error of one extraction (same as Ma's example implementation)
EPSILON = 2.0 ** -100

# How the seed is generated (None: the OS's cryptographic random numbers; integer: reproducible pseudorandom numbers)
SEED_RNG = None

# True: save the files of this run in a new folder results/toeplitz/<date_time>/; False: save nothing
SAVE_RESULTS = False
RESULTS_ROOT = Path("results/toeplitz")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if SAVE_RESULTS else None
print("save to    :", RUN_DIR if RUN_DIR is not None else "nothing is saved (SAVE_RESULTS = False)")

<!-- ## 2. 入力のビット列を用意する -->

## 2. Preparing the input bit string

<!-- C1 回路（1 量子ビットの重ね合わせ状態の測定）の出力を模したビット列を作ります。公平なビットに非対称な読み出し誤差 $P(1|0)$, $P(0|1)$ を加えたもので、1 の割合は $p_1 = \frac{1}{2}(1 - P(0|1)) + \frac{1}{2}P(1|0)$ です。`INPUT_PATH` を指定すると、そのファイルの raw data を使います。 -->

We create a bit string that mimics the output of the C1 circuit (measurement of a 1-qubit superposition). It consists of fair bits with asymmetric readout errors $P(1|0)$, $P(0|1)$ added, so the proportion of 1s is $p_1 = \frac{1}{2}(1 - P(0|1)) + \frac{1}{2}P(1|0)$. If `INPUT_PATH` is set, the raw data in that file is used.

<!-- 1 ビットあたりの min-entropy $h$ は、`H_MIN_OVERRIDE` を指定しない場合、SP 800-90B の Most Common Value（MCV）推定で決めます。 -->

If `H_MIN_OVERRIDE` is not set, the min-entropy per bit $h$ is determined by the Most Common Value (MCV) estimate of SP 800-90B.

$$
p_u = \min\left(1,\ \hat{p} + 2.576\sqrt{\frac{\hat{p}(1-\hat{p})}{L-1}}\right), \qquad h = -\log_2 p_u
$$

<!-- $\hat{p}$ は $L$ 個のサンプルのうち最も多い値の割合です。 -->

$\hat{p}$ is the proportion of the most common value among the $L$ samples.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `simulate_c1_bits` | 実機の raw data がないときの入力として、公平なビットに読み出し誤差によるビット反転を加え、C1 回路の測定結果を模擬したビット列を作ることを目的とした関数です。 | 長さ `n_bits` のビット列（`np.uint8` の配列） | `simulate_c1_bits(10, 0.02, 0.04, rng=1)` |
| `load_bits` | raw data などの 0 と 1 のテキストファイルを読み込み、抽出の入力に使える配列に変換することを目的とした関数です。 | ファイルのビット列（`np.uint8` の配列） | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |
| `save_bits` | ビット列を 0 と 1 の文字（改行なし）のテキストファイルとして、既存のファイルを上書きせずに保存することを目的とした関数です。 | 保存したパス | `save_bits("out/bits.txt", bits)` |
| `mcv_min_entropy` | 最頻値の出現割合の 99% 信頼上限 $p_u$ から、1 サンプルあたりの min-entropy $-\log_2 p_u$ を推定（MCV 推定）することを目的とした関数です。 | MCV 推定による 1 サンプルあたりの min-entropy（float） | `mcv_min_entropy(raw_bits)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `simulate_c1_bits` | Creates a bit string that mimics the measurement results of the C1 circuit by adding bit flips from readout errors to fair bits, as input when no raw data from real hardware is available. | Bit string of length `n_bits` (`np.uint8` array) | `simulate_c1_bits(10, 0.02, 0.04, rng=1)` |
| `load_bits` | Reads a text file of 0s and 1s, such as raw data, and converts it into an array that can be used as input for extraction. | Bit string of the file (`np.uint8` array) | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |
| `save_bits` | Saves a bit string as a text file of the characters 0 and 1 (no newlines), without overwriting an existing file. | Path of the saved file | `save_bits("out/bits.txt", bits)` |
| `mcv_min_entropy` | Estimates the min-entropy per sample $-\log_2 p_u$ from the 99% upper confidence bound $p_u$ of the proportion of the most common value (the MCV estimate). | Min-entropy per sample from the MCV estimate (float) | `mcv_min_entropy(raw_bits)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>n_bits</code></td><td>作るビット数</td><td>必須</td></tr>
<tr><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>rng</code></td><td>乱数シード（整数）、<code>np.random.Generator</code>、または <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>0 と 1 の文字だけのテキストファイル（空白・改行は無視）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>save_bits</code></td><td><code>path</code></td><td>保存先。既に存在する場合はエラー</td><td>必須</td></tr>
<tr><td><code>bits</code></td><td>0 と 1 の配列</td><td>必須</td></tr>
<tr><td><code>mcv_min_entropy</code></td><td><code>symbols</code></td><td>整数の配列。長さ 2 以上</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>n_bits</code></td><td>Number of bits to generate</td><td>Required</td></tr>
<tr><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>rng</code></td><td>Random seed (integer), <code>np.random.Generator</code>, or <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>Text file of only the characters 0 and 1 (whitespace and newlines are ignored)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>save_bits</code></td><td><code>path</code></td><td>Destination. An error is raised if it already exists</td><td>Required</td></tr>
<tr><td><code>bits</code></td><td>Array of 0s and 1s</td><td>Required</td></tr>
<tr><td><code>mcv_min_entropy</code></td><td><code>symbols</code></td><td>Array of integers. Length 2 or more</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def simulate_c1_bits(n_bits, p1_given_0=0.0, p0_given_1=0.0, rng=None):
    rng = np.random.default_rng(rng)
    ideal = rng.integers(0, 2, size=n_bits, dtype=np.uint8)        # ideal measurement results: fair bits
    u = rng.random(n_bits)
    flip = np.where(ideal == 0, u < p1_given_0, u < p0_given_1)    # flipped by readout errors
    return ideal ^ flip.astype(np.uint8)


def load_bits(path):
    text = "".join(Path(path).read_text(encoding="ascii").split())
    bits = np.frombuffer(text.encode("ascii"), dtype=np.uint8) - ord("0")
    if bits.size == 0 or np.any(bits > 1):
        raise ValueError(f"the file contains characters other than 0 and 1, or is empty: {path}")
    return bits.astype(np.uint8)


def save_bits(path, bits):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
    with path.open("x", encoding="ascii") as f:
        f.write((bits + ord("0")).tobytes().decode("ascii"))
    return path


def mcv_min_entropy(symbols):
    symbols = np.asarray(symbols).reshape(-1)
    L = len(symbols)
    _, counts = np.unique(symbols, return_counts=True)
    p_hat = counts.max() / L
    p_u = min(1.0, p_hat + 2.576 * math.sqrt(p_hat * (1 - p_hat) / (L - 1)))
    return -math.log2(p_u)

In [ ]:
N_BITS = 1_000_000
P1_GIVEN_0 = 0.02
P0_GIVEN_1 = 0.04
SIM_SEED = 1
INPUT_PATH = None       # e.g., Path("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")
H_MIN_OVERRIDE = None   # e.g., the result of the SP 800-90B assessment

raw_bits = simulate_c1_bits(N_BITS, P1_GIVEN_0, P0_GIVEN_1, rng=SIM_SEED) if INPUT_PATH is None else load_bits(INPUT_PATH)
h_min = mcv_min_entropy(raw_bits) if H_MIN_OVERRIDE is None else float(H_MIN_OVERRIDE)
print(f"Input: n = {raw_bits.size}, P(1) = {raw_bits.mean():.5f}")
print(f"h used for extraction = {h_min:.6f} ({'MCV estimate' if H_MIN_OVERRIDE is None else 'H_MIN_OVERRIDE'})")

<!-- ## 3. 出力の長さを決める -->

## 3. Determining the output length

<!-- 手法 B の $m = \lfloor nh + 2\log_2\varepsilon \rfloor$ を計算します。$m \le 0$ のとき（入力が短すぎるか、$h$ が小さすぎるとき）はエラーにします。 -->

We compute $m = \lfloor nh + 2\log_2\varepsilon \rfloor$ from Method B. If $m \le 0$ (the input is too short or $h$ is too small), an error is raised.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `calculate_output_length` | Leftover Hash Lemma にもとづいて、入力のビット数 $n$・min-entropy $h$・誤差 $\varepsilon$ から、抽出できる出力のビット数 $m$ を計算することを目的とした関数です。 | `(m, k)` のタプル。`m` は出力のビット数（int）、`k` は入力の min-entropy $nh$（float） | `calculate_output_length(4096, 6.7 / 8, 2.0 ** -100)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `calculate_output_length` | Computes the number of output bits $m$ that can be extracted from the number of input bits $n$, the min-entropy $h$, and the error $\varepsilon$, based on the Leftover Hash Lemma. | Tuple `(m, k)`. `m` is the number of output bits (int) and `k` is the min-entropy of the input $nh$ (float) | `calculate_output_length(4096, 6.7 / 8, 2.0 ** -100)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>calculate_output_length</code></td><td><code>input_length</code></td><td>入力のビット数 $n$</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>1 ビットあたりの min-entropy $h$（0〜1）</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>誤差 $\varepsilon$（0 < ε < 1）</td><td><code>2.0 ** -100</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>calculate_output_length</code></td><td><code>input_length</code></td><td>Number of input bits $n$</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>Min-entropy per bit $h$ (0 to 1)</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Error $\varepsilon$ (0 < ε < 1)</td><td><code>2.0 ** -100</code></td></tr>
</tbody>
</table>

In [ ]:
def calculate_output_length(input_length, min_entropy_per_bit, epsilon=2.0 ** -100):
    if input_length < 1:
        raise ValueError("input_length must be >= 1")
    if not (0.0 <= min_entropy_per_bit <= 1.0):
        raise ValueError("min_entropy_per_bit must be between 0 and 1")
    if not (0.0 < epsilon < 1.0):
        raise ValueError("epsilon must satisfy 0 < epsilon < 1")
    k = input_length * min_entropy_per_bit
    m = math.floor(k + 2.0 * math.log2(epsilon))    # solve epsilon = 2^((m-k)/2) for m
    if m <= 0:
        raise ValueError(f"output length m={m} <= 0. Increase the input length or epsilon.")
    return m, k

<!-- ## 4. シードを作る -->

## 4. Generating the seed

<!-- 手法 A の $n + m - 1$ ビットのシードを作り、Toeplitz 行列の 1 列目 $c$ と 1 行目 $r$ に分けます。 -->

We generate the $(n + m - 1)$-bit seed of Method A and split it into the first column $c$ and the first row $r$ of the Toeplitz matrix.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `generate_toeplitz_seed` | Toeplitz 行列を決める $n+m-1$ ビットのシードを生成することを目的とした関数です。 | 長さ $n+m-1$ のシード（`np.uint8` の配列） | `generate_toeplitz_seed(6, 3)` |
| `split_toeplitz_seed` | シードを Toeplitz 行列の 1 列目 $c$ と 1 行目 $r$ に分けることを目的とした関数です。 | `(first_column, first_row)` のタプル（$c$ と $r$、長さ $n$ と $m$） | `split_toeplitz_seed(seed, 6, 3)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `generate_toeplitz_seed` | Generates the $(n+m-1)$-bit seed that determines the Toeplitz matrix. | Seed of length $n+m-1$ (`np.uint8` array) | `generate_toeplitz_seed(6, 3)` |
| `split_toeplitz_seed` | Splits the seed into the first column $c$ and the first row $r$ of the Toeplitz matrix. | Tuple `(first_column, first_row)` ($c$ and $r$, of lengths $n$ and $m$) | `split_toeplitz_seed(seed, 6, 3)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>generate_toeplitz_seed</code></td><td><code>input_length</code></td><td>入力のビット数 $n$</td><td>必須</td></tr>
<tr><td><code>output_length</code></td><td>出力のビット数 $m$</td><td>必須</td></tr>
<tr><td><code>rng</code></td><td><code>None</code>（OS の暗号用乱数）、または乱数シードの整数・<code>np.random.Generator</code>（確認用）</td><td><code>None</code></td></tr>
<tr><td rowspan="3"><code>split_toeplitz_seed</code></td><td><code>seed</code></td><td>長さ $n+m-1$ のシード</td><td>必須</td></tr>
<tr><td><code>input_length</code></td><td>$n$</td><td>必須</td></tr>
<tr><td><code>output_length</code></td><td>$m$</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="3"><code>generate_toeplitz_seed</code></td><td><code>input_length</code></td><td>Number of input bits $n$</td><td>Required</td></tr>
<tr><td><code>output_length</code></td><td>Number of output bits $m$</td><td>Required</td></tr>
<tr><td><code>rng</code></td><td><code>None</code> (the OS's cryptographic random numbers), or an integer random seed or <code>np.random.Generator</code> (for checking)</td><td><code>None</code></td></tr>
<tr><td rowspan="3"><code>split_toeplitz_seed</code></td><td><code>seed</code></td><td>Seed of length $n+m-1$</td><td>Required</td></tr>
<tr><td><code>input_length</code></td><td>$n$</td><td>Required</td></tr>
<tr><td><code>output_length</code></td><td>$m$</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def generate_toeplitz_seed(input_length, output_length, rng=None):
    seed_length = input_length + output_length - 1
    if rng is None:
        random_bytes = secrets.token_bytes((seed_length + 7) // 8)
        seed = np.unpackbits(np.frombuffer(random_bytes, dtype=np.uint8))[:seed_length]
    else:
        seed = np.random.default_rng(rng).integers(0, 2, size=seed_length, dtype=np.uint8)
    return seed.astype(np.uint8, copy=False)


def split_toeplitz_seed(seed, input_length, output_length):
    seed = np.asarray(seed, dtype=np.uint8).reshape(-1)
    expected = input_length + output_length - 1
    if len(seed) != expected:
        raise ValueError(f"Seed length must be {expected}, got {len(seed)}")
    first_column = seed[:input_length]                 # c = (s_0, ..., s_{n-1})
    first_row = np.empty(output_length, dtype=np.uint8)
    first_row[0] = first_column[0]                     # r = (s_0, s_n, ..., s_{n+m-2})
    first_row[1:] = seed[input_length:]
    return first_column, first_row

<!-- ## 5. FFT で Toeplitz 行列との積を計算する -->

## 5. Computing the product with the Toeplitz matrix using the FFT

<!-- 手法 C の巡回畳み込みで $y = xT \bmod 2$ を計算します。 -->

We compute $y = xT \bmod 2$ with the cyclic convolution of Method C.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `toeplitz_hash_fft` | 入力 $x$ と Toeplitz 行列 $T$ の積 $y = xT \bmod 2$ を FFT で高速に計算し、$m$ ビットの出力を得ることを目的とした関数です。 | `(output_bits, seed)` のタプル。`output_bits` は長さ $m$、`seed` は長さ $n+m-1$ | `toeplitz_hash_fft(raw_bits, m, seed)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `toeplitz_hash_fft` | Computes the product $y = xT \bmod 2$ of the input $x$ and the Toeplitz matrix $T$ quickly with the FFT to obtain $m$ output bits. | Tuple `(output_bits, seed)`. `output_bits` has length $m$ and `seed` has length $n+m-1$ | `toeplitz_hash_fft(raw_bits, m, seed)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>toeplitz_hash_fft</code></td><td><code>raw_bits</code></td><td>入力のビット列 $x$（長さ $n$）</td><td>必須</td></tr>
<tr><td><code>output_length</code></td><td>出力のビット数 $m$（$1 \le m \le n$）</td><td>必須</td></tr>
<tr><td><code>seed</code></td><td>長さ $n+m-1$ のシード。<code>None</code> なら新しく作る</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td><code>seed=None</code> のときに <code>generate_toeplitz_seed</code> に渡す値</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>toeplitz_hash_fft</code></td><td><code>raw_bits</code></td><td>Input bit string $x$ (length $n$)</td><td>Required</td></tr>
<tr><td><code>output_length</code></td><td>Number of output bits $m$ ($1 \le m \le n$)</td><td>Required</td></tr>
<tr><td><code>seed</code></td><td>Seed of length $n+m-1$. <code>None</code> means generate a new one</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>Value passed to <code>generate_toeplitz_seed</code> when <code>seed=None</code></td><td><code>None</code></td></tr>
</tbody>
</table>

In [ ]:
def toeplitz_hash_fft(raw_bits, output_length, seed=None, rng=None):
    x = np.ascontiguousarray(raw_bits, dtype=np.uint8).reshape(-1)
    if np.any(x > 1):
        raise ValueError("raw_bits must contain only 0 and 1")
    n = len(x)
    m = int(output_length)
    if not (1 <= m <= n):
        raise ValueError("output_length must satisfy 1 <= m <= n")
    if seed is None:
        seed = generate_toeplitz_seed(n, m, rng)
    seed = np.asarray(seed, dtype=np.uint8).reshape(-1)
    first_column, first_row = split_toeplitz_seed(seed, n, m)

    # a = (r_0, ..., r_{m-1}, c_{n-1}, ..., c_1); pad x with m - 1 zeros at the end (length L = n + m - 1)
    L = n + m - 1
    kernel = np.concatenate((first_row, first_column[:0:-1])).astype(np.float64)
    padded_x = np.concatenate((x, np.zeros(m - 1, dtype=np.uint8))).astype(np.float64)
    convolution = np.fft.irfft(np.fft.rfft(kernel, n=L) * np.fft.rfft(padded_x, n=L), n=L)[:m]

    rounded = np.rint(convolution)
    max_error = float(np.max(np.abs(convolution - rounded)))
    if max_error > 0.25:
        raise ArithmeticError(f"FFT rounding error too large: {max_error}")
    output_bits = (rounded.astype(np.int64) % 2).astype(np.uint8)
    return output_bits, seed

<!-- ## 6. 抽出する -->

## 6. Extracting

<!-- ### 6.1 抽出の関数 -->

### 6.1 Extraction functions

<!-- `toeplitz_extraction` は入力全体を 1 つの Toeplitz 行列で、`toeplitz_extraction_blocks` は手法 D のとおりブロックごとに同じシードで抽出します。`toeplitz_extraction_blocks` は、最後の `block_size` に満たない部分を使いません。 -->

`toeplitz_extraction` extracts from the whole input with one Toeplitz matrix, and `toeplitz_extraction_blocks` extracts block by block with the same seed, as in Method D. `toeplitz_extraction_blocks` does not use the last part that is shorter than `block_size`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `toeplitz_extraction` | 出力の長さの決定からハッシュの計算までをまとめて行い、入力全体を 1 つの Toeplitz 行列で抽出することを目的とした関数です。 | `(output_bits, seed, result)` のタプル | `toeplitz_extraction(raw_bits, h_min, EPSILON)` |
| `toeplitz_extraction_blocks` | 入力を $n_b$ ビットのブロックに分け、すべてのブロックに同じシードを使い回して抽出し、出力を連結することを目的とした関数です。 | `(output_bits, seed, result)` のタプル | `toeplitz_extraction_blocks(raw_bits, 4096, h_min, EPSILON)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `toeplitz_extraction` | Performs everything from determining the output length to computing the hash, extracting from the whole input with one Toeplitz matrix. | Tuple `(output_bits, seed, result)` | `toeplitz_extraction(raw_bits, h_min, EPSILON)` |
| `toeplitz_extraction_blocks` | Splits the input into blocks of $n_b$ bits, extracts from every block reusing the same seed, and concatenates the outputs. | Tuple `(output_bits, seed, result)` | `toeplitz_extraction_blocks(raw_bits, 4096, h_min, EPSILON)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="5"><code>toeplitz_extraction</code></td><td><code>raw_bits</code></td><td>入力のビット列（長さ $n$）</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>1 ビットあたりの min-entropy $h$</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>誤差 $\varepsilon$</td><td><code>2.0 ** -100</code></td></tr>
<tr><td><code>seed</code></td><td>長さ $n+m-1$ のシード。<code>None</code> なら新しく作る</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td><code>seed=None</code> のときのシードの作り方（<code>generate_toeplitz_seed</code> と同じ）</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>toeplitz_extraction_blocks</code></td><td><code>raw_bits</code></td><td>入力のビット列</td><td>必須</td></tr>
<tr><td><code>block_size</code></td><td>1 ブロックのビット数 $n_b$</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>1 ビットあたりの min-entropy $h$</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>1 ブロックあたりの誤差 $\varepsilon$</td><td><code>2.0 ** -100</code></td></tr>
<tr><td><code>seed</code></td><td>長さ $n_b+m_b-1$ のシード。<code>None</code> なら新しく作る</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td><code>seed=None</code> のときのシードの作り方</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="5"><code>toeplitz_extraction</code></td><td><code>raw_bits</code></td><td>Input bit string (length $n$)</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>Min-entropy per bit $h$</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Error $\varepsilon$</td><td><code>2.0 ** -100</code></td></tr>
<tr><td><code>seed</code></td><td>Seed of length $n+m-1$. <code>None</code> means generate a new one</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>How the seed is generated when <code>seed=None</code> (same as <code>generate_toeplitz_seed</code>)</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>toeplitz_extraction_blocks</code></td><td><code>raw_bits</code></td><td>Input bit string</td><td>Required</td></tr>
<tr><td><code>block_size</code></td><td>Number of bits per block $n_b$</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>Min-entropy per bit $h$</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Error per block $\varepsilon$</td><td><code>2.0 ** -100</code></td></tr>
<tr><td><code>seed</code></td><td>Seed of length $n_b+m_b-1$. <code>None</code> means generate a new one</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>How the seed is generated when <code>seed=None</code></td><td><code>None</code></td></tr>
</tbody>
</table>

<br>

<!-- 戻り値の `result` の主なキーは次のとおりです。 -->

The main keys of the returned `result` are as follows.

<!--
| キー | 意味 |
|---|---|
| `input_length_n`, `min_entropy_rate`, `total_min_entropy_k` | $n$, $h$, $k = nh$（`toeplitz_extraction`） |
| `output_length_m`, `seed_length` | 出力のビット数 $m$、シードのビット数 $n + m - 1$（`toeplitz_extraction`） |
| `block_size`, `n_blocks`, `unused_bits` | $n_b$、$N_b$、使わなかった入力のビット数（`toeplitz_extraction_blocks`） |
| `output_length_per_block`, `total_output_length` | $m_b$ と $N_b m_b$（`toeplitz_extraction_blocks`） |
| `log2_epsilon` / `log2_total_epsilon` | 誤差 $\varepsilon$ / 全体の誤差 $N_b\varepsilon$ の $\log_2$ |
-->

| Key | Meaning |
|---|---|
| `input_length_n`, `min_entropy_rate`, `total_min_entropy_k` | $n$, $h$, $k = nh$ (`toeplitz_extraction`) |
| `output_length_m`, `seed_length` | Number of output bits $m$ and number of seed bits $n + m - 1$ (`toeplitz_extraction`) |
| `block_size`, `n_blocks`, `unused_bits` | $n_b$, $N_b$, and the number of unused input bits (`toeplitz_extraction_blocks`) |
| `output_length_per_block`, `total_output_length` | $m_b$ and $N_b m_b$ (`toeplitz_extraction_blocks`) |
| `log2_epsilon` / `log2_total_epsilon` | $\log_2$ of the error $\varepsilon$ / of the total error $N_b\varepsilon$ |

In [ ]:
def toeplitz_extraction(raw_bits, min_entropy_per_bit, epsilon=2.0 ** -100, seed=None, rng=None):
    raw_bits = np.asarray(raw_bits, dtype=np.uint8).reshape(-1)
    n = len(raw_bits)
    m, k = calculate_output_length(n, min_entropy_per_bit, epsilon)
    output_bits, seed = toeplitz_hash_fft(raw_bits, m, seed=seed, rng=rng)
    result = {
        "input_length_n": n,
        "min_entropy_rate": float(min_entropy_per_bit),
        "total_min_entropy_k": float(k),
        "epsilon": float(epsilon),
        "log2_epsilon": math.log2(epsilon),
        "output_length_m": int(m),
        "seed_length": int(len(seed)),
    }
    return output_bits, seed, result


def toeplitz_extraction_blocks(raw_bits, block_size, min_entropy_per_bit, epsilon=2.0 ** -100, seed=None, rng=None):
    raw_bits = np.asarray(raw_bits, dtype=np.uint8).reshape(-1)
    n_blocks = len(raw_bits) // block_size
    if n_blocks == 0:
        raise ValueError("raw_bits is shorter than block_size")
    m, k = calculate_output_length(block_size, min_entropy_per_bit, epsilon)
    if seed is None:
        seed = generate_toeplitz_seed(block_size, m, rng)
    # use the same seed (the same Toeplitz matrix) for every block
    blocks = raw_bits[:n_blocks * block_size].reshape(n_blocks, block_size)
    output_bits = np.concatenate([toeplitz_hash_fft(block, m, seed)[0] for block in blocks])
    total_epsilon = n_blocks * epsilon
    result = {
        "block_size": int(block_size),
        "n_blocks": int(n_blocks),
        "unused_bits": int(len(raw_bits) - n_blocks * block_size),
        "min_entropy_rate": float(min_entropy_per_bit),
        "epsilon": float(epsilon),
        "output_length_per_block": int(m),
        "total_output_length": int(n_blocks * m),
        "seed_length": int(len(seed)),
        "total_epsilon": total_epsilon,
        "log2_total_epsilon": math.log2(total_epsilon),
    }
    return output_bits, seed, result

<!-- ### 6.2 抽出を実行する -->

### 6.2 Running the extraction

<!-- 2. の入力を抽出します。`BLOCK_SIZE = None` なら入力全体を 1 つの行列で、整数なら `BLOCK_SIZE` ビットごとに同じシードで抽出します（Ma と同じにするには `4096`）。 -->

We extract from the input of 2. If `BLOCK_SIZE = None`, the whole input is extracted with one matrix; if it is an integer, every `BLOCK_SIZE` bits are extracted with the same seed (use `4096` to match Ma).

In [ ]:
BLOCK_SIZE = None   # e.g., 4096

started = time.perf_counter()
if BLOCK_SIZE is None:
    extracted, toeplitz_seed, extraction_result = toeplitz_extraction(raw_bits, h_min, EPSILON, rng=SEED_RNG)
else:
    extracted, toeplitz_seed, extraction_result = toeplitz_extraction_blocks(raw_bits, BLOCK_SIZE, h_min, EPSILON, rng=SEED_RNG)
elapsed = time.perf_counter() - started

for key, value in extraction_result.items():
    print(f"{key:24s}: {value}")
print(f"Computation time = {elapsed:.2f} s")
print(f"Output: {extracted.size} bits, P(1) = {extracted.mean():.5f}")
print("First 40 bits of the output:", "".join(map(str, extracted[:40])))

<!-- ## 7. 抽出結果を保存する -->

## 7. Saving the extraction results

<!-- 1. で `SAVE_RESULTS = True` にした場合は、`RUN_DIR` の次のファイルに保存します（既存のファイルは上書きしません）。 -->

If `SAVE_RESULTS = True` in 1., the results are saved to the following files in `RUN_DIR` (existing files are not overwritten).

<!--
- `input_bits.txt`：抽出の入力のビット列（0 と 1 の文字、改行なし）
- `toeplitz_extracted.txt`：抽出したビット列（0 と 1 の文字、改行なし。`load_bits` で読み込めます）
- `toeplitz_seed.txt`：シード（同じ形式）
- `toeplitz_info.json`：`extraction_result` と入力の情報
-->

- `input_bits.txt`: the input bit string of the extraction (the characters 0 and 1, no newlines)
- `toeplitz_extracted.txt`: the extracted bit string (the characters 0 and 1, no newlines; can be read with `load_bits`)
- `toeplitz_seed.txt`: the seed (same format)
- `toeplitz_info.json`: `extraction_result` and information about the input

<!-- 同じ入力・$h$・$\varepsilon$・`BLOCK_SIZE` とこのシードを 6.1 の関数に渡すと、同じ出力が得られます。 -->

Passing the same input, $h$, $\varepsilon$, `BLOCK_SIZE`, and this seed to the functions in 6.1 gives the same output.

In [ ]:
if RUN_DIR is None:
    print("SAVE_RESULTS = False, so nothing is saved.")
else:
    # Existing files are never overwritten
    save_bits(RUN_DIR / "input_bits.txt", raw_bits)
    save_bits(RUN_DIR / "toeplitz_extracted.txt", extracted)
    save_bits(RUN_DIR / "toeplitz_seed.txt", toeplitz_seed)
    info = dict(extraction_result, input_path=None if INPUT_PATH is None else str(INPUT_PATH),
                simulation=None if INPUT_PATH is not None else dict(
                    n_bits=N_BITS, p1_given_0=P1_GIVEN_0, p0_given_1=P0_GIVEN_1, seed=SIM_SEED),
                h_min_source="MCV" if H_MIN_OVERRIDE is None else "H_MIN_OVERRIDE", block_size=BLOCK_SIZE)
    with (RUN_DIR / "toeplitz_info.json").open("x", encoding="utf-8") as f:
        f.write(json.dumps(info, ensure_ascii=False, indent=2) + "\n")
    print("Saved to:", RUN_DIR)